In [ ]:
from TerraFin import configure, load_terrafin_config
from TerraFin.data import DataFactory
from TerraFin.data.utils.filters import date_filter


configure()
config = load_terrafin_config()

test_data = DataFactory().get("NVDA")
test_data = date_filter(test_data, "2024-01-01", "2025-6-30")

### Base EDA

In [ ]:
# Check normality of return

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from scipy.stats import norm

from TerraFin.analytics.analysis.base_analytics import get_returns


returns = get_returns(test_data)

sns.histplot(returns, kde=False, bins=50, stat="density")

# calculate probability density function (PDF)
x = np.linspace(returns.min(), returns.max(), 1000)
y = norm.pdf(x, returns.mean(), returns.std())

# plot normal distribution line
sns.lineplot(x=x, y=y, color="red")
plt.show()

### Option Analysis

In [ ]:
# GEX Analysis
from TerraFin.analytics.analysis.options.gamma_exposure import get_current_gex


gex_output = get_current_gex("NVDA")
gex_output.gex_by_expiration.show()
gex_output.gex_by_strike.show()

### LPPL (Log-Periodic Power Law) Bubble Detection

Based on Didier Sornette's complexity economics framework.
Detects super-exponential growth with accelerating log-periodic oscillations
that often appear before crowded speculative peaks.

**Notebook note:**
- `lppl(closes)` uses TerraFin's calibrated default scan so notebook results match the chart and agent API
- `lppl(closes, n_windows=None)` switches to the full article-style 750→50 ladder for deeper research/debug runs

**Caveats:**
- Best suited for market indices / asset classes, not individual stocks
- Never use in isolation — combine with macro indicators (rates, valuations, liquidity)
- High confidence is a fragility warning, not a crash-timing oracle


In [ ]:
# LPPL Bubble Detection
import matplotlib.pyplot as plt
import numpy as np

from TerraFin.analytics.analysis.technical.lppl import lppl
from TerraFin.data import DataFactory
from TerraFin.data.utils.filters import date_filter


# # Example LPPL window on Nasdaq
# df = DataFactory().get("Nasdaq")
# df = date_filter(df, "1998-02-14", "1999-12-31")

# # Alternative example on KOSPI
# df = DataFactory().get("Kospi")
# df = date_filter(df, "2019-01-16", "2021-01-05")

# Recent KOSPI
df = DataFactory().get("Kospi")
df = date_filter(df, "2025-01-01", "2026-03-01")

# Default call: calibrated scan used by TerraFin charts and agent/API flows
closes = df["close"].astype(float).tolist()
result = lppl(closes)

# Optional research mode: full article ladder (750 -> 50 by 5 trading days)
# article_result = lppl(closes, n_windows=None)

# Confidence interpretation (per Sornette / FCO)
#   0-5%:   No LPPL pattern (normal market)
#   5-15%:  Some bubble possibility (caution)
#   15-25%: Growing bubble formation (warning)
#   25-40%: High confidence (overheated)
#   40%+:   Very high confidence (bubble peak)
print(f"LPPL Bubble Confidence: {result.confidence * 100:.1f}%")
print(f"Qualifying fits: {len(result.qualifying_fits)} / {result.total_windows} windows\n")

for i, qf in enumerate(result.qualifying_fits):
    c_over_b = qf.c / abs(qf.b) if abs(qf.b) > 1e-12 else 0.0
    print(f"  Window {i + 1}: tc={qf.tc:.0f}, m={qf.m:.3f}, omega={qf.omega:.2f}, |C/B|={c_over_b:.3f}")

# Plot full-series fit
if result.fit:
    fig, ax = plt.subplots(figsize=(12, 5))
    log_prices = np.log(closes)
    ax.plot(log_prices, label="ln(Nasdaq)", alpha=0.7)
    ax.plot(result.fit.fitted, label="LPPL Fit", linestyle="--", color="red")
    ax.set_xlabel("Trading Days")
    ax.set_ylabel("Log Price")
    ax.set_title(f"LPPL Model Fit — Nasdaq (Bubble Confidence: {result.confidence * 100:.1f}%)")
    ax.legend()
    plt.tight_layout()
    plt.show()
